# Week 79, Day 1: Secure coding

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w079d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from pathlib import Path

base = Path("/srv/files").resolve()
for name in ["report.pdf", "../etc/passwd", "photos/../report.pdf", "/etc/shadow"]:
    target = (base / name).resolve()
    print(f"{name:<22}{target.as_posix():<22}{target.is_relative_to(base)}")

In [ ]:
import re

ALLOWED = {"username", "email", "age"}

def validate(data):
    errors = {}
    unknown = set(data) - ALLOWED
    if unknown:
        errors["_"] = f"unexpected fields: {', '.join(sorted(unknown))}"
    name = str(data.get("username", "")).strip()
    if not re.fullmatch(r"[A-Za-z0-9_]{3,20}", name):
        errors["username"] = "3 to 20 letters, digits or underscores"
    age = data.get("age")
    if not isinstance(age, int) or isinstance(age, bool) or not 13 <= age <= 120:
        errors["age"] = "a whole number from 13 to 120"
    return errors

print(validate({"username": "ada_l", "email": "a@x.org", "age": 36}))
print(validate({"username": "ada; DROP TABLE", "age": "36", "is_admin": True}))
print(re.match(r"[a-z]+", "ok<script>") is not None, re.fullmatch(r"[a-z]+", "ok<script>") is not None)

In [ ]:
import ast, json

print(json.loads('{"size": 3, "tags": ["a", "b"]}'))
print(ast.literal_eval("[1, 2.5, 'three', {'four': (5, None)}]"))
for attack in ("__import__('os').system('rm -rf /')", "(1).__class__"):
    try:
        ast.literal_eval(attack)
    except ValueError as e:
        print("refused:", type(e).__name__)

> **Key idea.** Model threats: assets, attackers, and trust boundaries where untrusted data enters (requests, uploads and their names, other services, files). Apply least privilege, defence in depth, secure defaults, fail closed, and simplicity; the OWASP Top 10 is a review checklist. Validate at the boundary with allow-lists: types (bool is an int!), ranges and lengths both ways, formats with re.fullmatch, choices, unknown fields refused (mass assignment), normalised first. Never let untrusted data choose code or files: resolve paths and check is_relative_to; json or ast.literal_eval, never eval or pickle; yaml.safe_load; limits on size and time. Fail closed; show generic errors with an incident id and log details privately; redact secrets from logs; compare secrets with hmac.compare_digest; rate-limit guessing and give one message for any login failure. Bandit and Ruff's S rules find many mistakes automatically.

### Exercise: Stay inside the folder

Write `safe_path(base, name)`: the resolved path of `name` inside the folder `base`, or raise `PermissionError(f"outside {BASE_NAME}: {name}")` (BASE_NAME is the folder's own name) if it would land anywhere else, including the folder itself. Also refuse, with the same error, any name containing a NUL character (`"\0"`), and an empty name. Then `read_user_file(base, name)`: the text of that file, using `safe_path` (a missing file raises `FileNotFoundError` as usual).

In [ ]:
from pathlib import Path

def safe_path(base, name):
    ___

def read_user_file(base, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_inside():\n    \"names inside the folder\"\n    from pathlib import Path\n    Path(\"files/sub\").mkdir(parents=True, exist_ok=True)\n    Path(\"files/a.txt\").write_text(\"hello\")\n    Path(\"files/sub/b.txt\").write_text(\"deep\")\n    assert safe_path(\"files\", \"a.txt\") == Path(\"files/a.txt\").resolve()\n    assert read_user_file(\"files\", \"sub/../a.txt\") == \"hello\" and read_user_file(\"files\", \"sub/b.txt\") == \"deep\"\n\ndef test_outside():\n    \"traversal, absolute paths, NUL and the folder itself\"\n    from pathlib import Path\n    Path(\"files\").mkdir(exist_ok=True)\n    Path(\"secret.txt\").write_text(\"top secret\")\n    for name in [\"../secret.txt\", \"sub/../../secret.txt\", \"/etc/passwd\", \"\", \".\", \"sub/..\", \"a.txt\\0.png\"]:\n        try:\n            safe_path(\"files\", name)\n        except PermissionError as e:\n            assert str(e) == f\"outside files: {name}\", name\n        else:\n            raise AssertionError(f\"{name!r} must be refused\")\n    Path(\"files2\").mkdir(exist_ok=True)\n    Path(\"files2/x.txt\").write_text(\"next door\")\n    try:\n        read_user_file(\"files\", \"../files2/x.txt\")\n    except PermissionError:\n        pass\n    else:\n        raise AssertionError(\"a sibling folder with a similar name is still outside\")")

### Exercise: Allow-list validation

Write `validate_signup(data)` for a JSON sign-up body. Return `(clean, errors)`: `errors` maps field names to messages (empty if all is well), and `clean` holds the normalised values only when there are no errors (else `None`). Rules: only the fields `username`, `email`, `password` and `age` are allowed (any other field: error under `"_"`, `"unexpected field(s): A, B"` sorted); `username`: a string, stripped and normalised with `unicodedata.normalize("NFKC", ...)`, then 3 to 20 characters of `A-Z`, `a-z`, `0-9` and `_` (`"3 to 20 letters, digits or underscores"`); `email`: a string, stripped and lower-cased, at most 254 characters, matching `[^@\s]+@[^@\s]+\.[a-z]{2,}` in full (`"not an email address"`); `password`: a string of 12 to 128 characters (`"12 to 128 characters"`; never put the password in a message); `age`: optional, but if present an `int` (not a `bool`) from 13 to 120 (`"a whole number from 13 to 120"`). A missing required field gets `"required"`.

In [ ]:
import re
import unicodedata

ALLOWED = {"username", "email", "password", "age"}

def validate_signup(data):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_good():\n    \"normalised clean values\"\n    clean, errors = validate_signup({\"username\": \" ada_l \", \"email\": \" Ada@Example.ORG\", \"password\": \"correct horse battery\", \"age\": 36})\n    assert errors == {} and clean == {\"username\": \"ada_l\", \"email\": \"ada@example.org\", \"password\": \"correct horse battery\", \"age\": 36}\n    clean, errors = validate_signup({\"username\": \"\uff47\uff52\uff41\uff43\uff45\", \"email\": \"g@navy.mil\", \"password\": \"x\" * 12})\n    assert errors == {} and clean[\"username\"] == \"grace\" and \"age\" not in clean\n\ndef test_bad():\n    \"every rule\"\n    clean, errors = validate_signup({\"username\": \"a;b\", \"email\": \"nope@\", \"password\": \"short\", \"age\": True, \"is_admin\": True, \"role\": \"x\"})\n    assert clean is None\n    assert errors == {\"_\": \"unexpected field(s): is_admin, role\", \"username\": \"3 to 20 letters, digits or underscores\",\n                      \"email\": \"not an email address\", \"password\": \"12 to 128 characters\", \"age\": \"a whole number from 13 to 120\"}\n    clean, errors = validate_signup({\"email\": 42, \"password\": None, \"age\": \"36\"})\n    assert errors == {\"username\": \"required\", \"email\": \"not an email address\", \"password\": \"12 to 128 characters\",\n                      \"age\": \"a whole number from 13 to 120\"}\n    assert \"x\" * 300 not in str(validate_signup({\"username\": \"x\" * 300, \"email\": \"a@b.co\", \"password\": \"y\" * 300})[1])\n    assert validate_signup({\"username\": \"abc\", \"email\": \"a\" * 250 + \"@b.co\", \"password\": \"p\" * 12})[1] == {\"email\": \"not an email address\"}\n    assert validate_signup({\"username\": \"abc\", \"email\": \"a@b.co\\nBcc: x@y.co\", \"password\": \"p\" * 12})[1] == {\"email\": \"not an email address\"}")

### Exercise: Data, not code

A settings file stores values written like Python literals (`port = 8080`, `hosts = ["a", "b"]`, `debug = False`), and the old code read them with `eval`. Write `parse_settings(text)`: for each non-blank line not starting with `#`, split it at the first `=` into a name (stripped, matching `[a-z_][a-z0-9_]*` in full) and a value read with `ast.literal_eval`. Return a dict. Raise `ValueError(f"line N: bad setting")` for a line without `=` or with a bad name, and `ValueError(f"line N: not a plain value")` when `literal_eval` refuses the value (it raises `ValueError` or `SyntaxError`), counting lines from 1. Nothing in the file may ever run.

In [ ]:
import ast
import re

def parse_settings(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"literals only\"\n    text = \"# settings\\nport = 8080\\nhosts = ['a.example', 'b.example']\\n\\ndebug = False\\nlimits = {'upload': 10 * 1024}\\nratio = 0.75\\n\"\n    try:\n        got = parse_settings(text)\n    except ValueError as e:\n        assert str(e) == \"line 6: not a plain value\"\n    else:\n        raise AssertionError(\"10 * 1024 is an expression, not a plain value\")\n    got = parse_settings(text.replace(\"10 * 1024\", \"10240\"))\n    assert got == {\"port\": 8080, \"hosts\": [\"a.example\", \"b.example\"], \"debug\": False, \"limits\": {\"upload\": 10240}, \"ratio\": 0.75}\n    assert parse_settings(\"name = 'a = b'\") == {\"name\": \"a = b\"}\n\ndef test_attacks():\n    \"nothing runs\"\n    import builtins\n    ran = []\n    builtins._atlas_spy = ran.append\n    for line, message in [(\"x = __import__('os').getcwd()\", \"line 1: not a plain value\"),\n                          (\"x = _atlas_spy(1)\", \"line 1: not a plain value\"),\n                          (\"x = [i for i in range(3)]\", \"line 1: not a plain value\"),\n                          (\"Bad-Name = 1\", \"line 1: bad setting\"), (\"just text\", \"line 1: bad setting\"),\n                          (\"ok = 1\\nx = (1,\", \"line 2: not a plain value\")]:\n        try:\n            parse_settings(line)\n        except ValueError as e:\n            assert str(e) == message, (line, str(e))\n        else:\n            raise AssertionError(line)\n    del builtins._atlas_spy\n    assert ran == []\n    assert \"eval(\" not in source.replace(\"literal_eval(\", \"\"), \"don't use eval\"")

### Exercise: Logs without secrets

Write `redact(value)` for log records: it returns a copy with secrets hidden, going into dicts, lists and tuples at any depth. A dict key whose lower-case name contains `password`, `secret`, `token`, `apikey`, `api_key` or `authorization`, or is exactly `cookie` or `set-cookie`, has its value replaced by `"[REDACTED]"` (whatever the value is). In every string, wherever it is: replace each run of 13 to 19 digits (optionally separated by single spaces or hyphens) that looks like a card number with `"[CARD]"`, and replace `Bearer ` followed by a token (letters, digits and `-._~+/=`) with `"Bearer [REDACTED]"`. Other values are returned unchanged. The original must not be modified.

In [ ]:
import re

SECRET_WORDS = ("password", "secret", "token", "apikey", "api_key", "authorization")

def redact(value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_keys():\n    \"secret fields at any depth\"\n    record = {\"user\": \"ada\", \"password\": \"hunter2\", \"headers\": {\"Authorization\": \"Bearer abc.def\", \"Accept\": \"json\"},\n              \"items\": [{\"api_key\": 123}, {\"name\": \"x\", \"refresh_token\": None}], \"Cookie\": \"s=1\", \"pair\": (\"ok\", {\"Secret\": \"s\"})}\n    got = redact(record)\n    assert got == {\"user\": \"ada\", \"password\": \"[REDACTED]\", \"headers\": {\"Authorization\": \"[REDACTED]\", \"Accept\": \"json\"},\n                   \"items\": [{\"api_key\": \"[REDACTED]\"}, {\"name\": \"x\", \"refresh_token\": \"[REDACTED]\"}], \"Cookie\": \"[REDACTED]\",\n                   \"pair\": (\"ok\", {\"Secret\": \"[REDACTED]\"})}\n    assert record[\"password\"] == \"hunter2\" and record[\"headers\"][\"Authorization\"] == \"Bearer abc.def\"\n\ndef test_strings():\n    \"cards and bearer tokens inside text\"\n    assert redact(\"paid with 4111 1111 1111 1111 today\") == \"paid with [CARD] today\"\n    assert redact(\"card 4111-1111-1111-1111, ref 12345\") == \"card [CARD], ref 12345\"\n    assert redact(\"order 2026111012345\") == \"order [CARD]\"\n    assert redact(\"call 0161 496 0000\") == \"call 0161 496 0000\"\n    assert redact([\"sent Bearer eyJhbGc.eyJzdWIi.sig_-x to /api\", 42, None]) == [\"sent Bearer [REDACTED] to /api\", 42, None]")

### Exercise: Errors that don't leak

Write `handle(func, log, debug=False, new_id=lambda: secrets.token_hex(4))`: call `func()` and return `(200, result)`. If it raises `PermissionError`, return `(403, {"error": "not allowed"})`. If it raises `KeyError` (something wasn't found), return `(404, {"error": "not found"})`. Any other exception is unexpected: make an incident id with `new_id()`, call `log(f"incident ID: TYPE: MESSAGE")` with the exception's class name and message, and return `(500, {"error": "something went wrong", "incident": ID})`, adding `"detail": "TYPE: MESSAGE"` only when `debug` is true. The 500 body must never contain the exception's message otherwise.

In [ ]:
import secrets

def handle(func, log, debug=False, new_id=lambda: secrets.token_hex(4)):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_handle():\n    \"expected and unexpected errors\"\n    logged = []\n    assert handle(lambda: {\"ok\": 1}, logged.append) == (200, {\"ok\": 1})\n    def denied():\n        raise PermissionError(\"user 7 is not an admin\")\n    def missing():\n        raise KeyError(\"order 9\")\n    def broken():\n        raise ConnectionError(\"postgres://app:S3cret@db:5432/main unreachable\")\n    assert handle(denied, logged.append) == (403, {\"error\": \"not allowed\"})\n    assert handle(missing, logged.append) == (404, {\"error\": \"not found\"})\n    status, body = handle(broken, logged.append, new_id=lambda: \"ab12cd34\")\n    assert (status, body) == (500, {\"error\": \"something went wrong\", \"incident\": \"ab12cd34\"})\n    assert \"S3cret\" not in str(body)\n    assert logged == [\"incident ab12cd34: ConnectionError: postgres://app:S3cret@db:5432/main unreachable\"]\n    status, body = handle(broken, logged.append, debug=True, new_id=lambda: \"x\")\n    assert body[\"detail\"] == \"ConnectionError: postgres://app:S3cret@db:5432/main unreachable\"\n    status, body = handle(broken, [].append)\n    assert len(body[\"incident\"]) == 8")

### Exercise: Slowing down guessing

Write a class `LoginGuard(users, clock, max_failures=5, window=900)`: `users` maps usernames to their secret token (a string). `login(username, token, ip)` returns `"ok"`, `"denied"` or `"locked"`. Failures are counted per username **and** per IP address over the last `window` seconds; if either has reached `max_failures` failures in the window, return `"locked"` without checking anything. Compare tokens with `hmac.compare_digest` (on the encoded bytes); an unknown username is checked against a dummy token, so it takes the same path and returns the same `"denied"`. A success clears that username's failures (not the IP's). Failures older than the window no longer count.

In [ ]:
import hmac

class LoginGuard:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_guard():\n    \"lockout per user and per address\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    g = LoginGuard({\"ada\": \"tok-ada\", \"bob\": \"tok-bob\"}, clock)\n    assert g.login(\"ada\", \"tok-ada\", \"1.1.1.1\") == \"ok\"\n    for i in range(5):\n        assert g.login(\"ada\", f\"guess{i}\", f\"2.2.2.{i}\") == \"denied\"\n    assert g.login(\"ada\", \"tok-ada\", \"3.3.3.3\") == \"locked\"\n    assert g.login(\"bob\", \"tok-bob\", \"3.3.3.3\") == \"ok\"\n    clock.t = 901\n    assert g.login(\"ada\", \"tok-ada\", \"3.3.3.3\") == \"ok\"\n    for name in (\"x1\", \"x2\", \"x3\", \"x4\", \"ghost\"):\n        assert g.login(name, \"nope\", \"9.9.9.9\") == \"denied\"\n    assert g.login(\"bob\", \"tok-bob\", \"9.9.9.9\") == \"locked\"\n    assert g.login(\"bob\", \"tok-bob\", \"8.8.8.8\") == \"ok\"\n\ndef test_success_resets_user():\n    \"a success clears that user's count\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    g = LoginGuard({\"ada\": \"t\"}, Clock(), max_failures=3)\n    g.login(\"ada\", \"a\", \"1.1.1.1\")\n    g.login(\"ada\", \"b\", \"1.1.1.2\")\n    assert g.login(\"ada\", \"t\", \"1.1.1.3\") == \"ok\"\n    assert g.login(\"ada\", \"c\", \"1.1.1.4\") == \"denied\" and g.login(\"ada\", \"d\", \"1.1.1.5\") == \"denied\"\n    assert g.login(\"ada\", \"t\", \"1.1.1.6\") == \"ok\"\n    assert \"compare_digest\" in source\n\ndef test_constant_time_path():\n    \"unknown users take the same path\"\n    import hmac\n    real, calls = hmac.compare_digest, []\n    def spy(a, b):\n        calls.append(1)\n        return real(a, b)\n    hmac.compare_digest = spy\n    try:\n        class Clock:\n            t = 0\n            def __call__(self):\n                return self.t\n        g = LoginGuard({\"ada\": \"t\"}, Clock())\n        assert g.login(\"nobody\", \"t\", \"1.1.1.1\") == \"denied\" and g.login(\"ada\", \"x\", \"1.1.1.1\") == \"denied\"\n    finally:\n        hmac.compare_digest = real\n    assert len(calls) == 2")

### Bug hunt: A check that a neighbour passed

The check compares the start of the path as a **string**, so `/srv/data-private` counts as inside `/srv/data`, and this prints `True True`. Check real path containment, so it prints `True False`.

In [ ]:
from pathlib import PurePosixPath

BASE = PurePosixPath("/srv/data")

def inside(path):
    return str(path).startswith(str(BASE))

print(inside(PurePosixPath("/srv/data/report.csv")), inside(PurePosixPath("/srv/data-private/keys.txt")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"is_relative_to\"\n    expect_output(\"True False\")")

### Bug hunt: Failing open

When the permissions service is down, `can_delete` raises, and the `except` lets everyone through, so this prints `True`. Fail closed: when the check can't be completed, the answer is no, so it prints `False`.

In [ ]:
def lookup_role(user):
    raise ConnectionError("permissions service unavailable")

def can_delete(user):
    try:
        return lookup_role(user) == "admin"
    except Exception:
        return True   # don't block people when the service is down

print(can_delete("mallory"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"deny when unsure\"\n    expect_output(\"False\")")

### Bug hunt: A validator that accepted extra

`re.match` only checks that the **start** of the text matches, so a username followed by anything at all passes, and this prints `['ada', 'ada<script>', 'ada; DROP TABLE users']`. Require the whole value to match, so it prints `['ada']`.

In [ ]:
import re

names = ["ada", "ada<script>", "ada; DROP TABLE users", "a"]
print([n for n in names if re.match(r"[a-z_]{3,20}", n)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fullmatch\"\n    expect_output(\"['ada']\")")

### Puzzle: Serving a user's file safely

Arrange a function that serves a file a user asked for by name, refusing anything outside the uploads folder. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # isinstance(True, int)
q = ___   # (Path("/srv/a") / "/etc/x").as_posix()
r = ___   # "allow-list" or "deny-list": the safer way to validate
s = ___   # the function that safely reads a Python literal from text (module.function)
t = ___   # True or False: a failed permission check should allow access

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"/etc/x\" and r == \"allow-list\", \"p, q and r\"\n    assert s == \"ast.literal_eval\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a safe download endpoint

Write `download(request, sessions, owners, base, log)` for a file-download endpoint, returning `(status, headers, body)`. `request` is a dict with `"cookies"` (a dict) and `"query"` (a dict); `sessions` maps session ids to usernames; `owners` maps file names (relative paths inside `base`) to the username that owns them. Rules, in order: no `session` cookie, or one not in `sessions` (compare each known id with `hmac.compare_digest`), gives `401` with body `b"log in first"`. A missing or empty `file` query value gives `400` (`b"which file?"`). A name that resolves outside `base` (use your `safe_path` idea: resolve, then `is_relative_to`, refusing the folder itself, empty names and NUL characters) gives `403` (`b"not allowed"`) and calls `log(f"blocked traversal by USER: NAME!r")` with the name in `repr` form, so newlines in it can't forge log lines. A file not in `owners` or not owned by this user gives `404` (`b"not found"`), the **same** answer either way, so users can't discover other people's files. A file that's listed but missing on disk also gives `404`. Otherwise `200`, the file's bytes, and headers `Content-Type: application/octet-stream`, `X-Content-Type-Options: nosniff`, and `Content-Disposition: attachment; filename="NAME"`, where NAME is the file's own name with every character other than letters, digits, `.`, `-` and `_` replaced by `_`.

In [ ]:
import hmac
import re
from pathlib import Path

def download(request, sessions, owners, base, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _setup():\n    import shutil\n    from pathlib import Path\n    shutil.rmtree(\"store\", ignore_errors=True)\n    Path(\"store/ada\").mkdir(parents=True)\n    Path(\"store/bob\").mkdir(parents=True)\n    Path(\"store/ada/report 2026 (final).pdf\").write_bytes(b\"%PDF ada\")\n    Path(\"store/bob/notes.txt\").write_bytes(b\"bob's notes\")\n    Path(\"secret.env\").write_text(\"KEY=1\")\n    sessions = {\"s-ada-123\": \"ada\", \"s-bob-456\": \"bob\"}\n    owners = {\"ada/report 2026 (final).pdf\": \"ada\", \"bob/notes.txt\": \"bob\", \"ada/missing.txt\": \"ada\"}\n    return sessions, owners\n\ndef _req(session=None, file=None):\n    return {\"cookies\": {} if session is None else {\"session\": session}, \"query\": {} if file is None else {\"file\": file}}\n\ndef test_success():\n    \"the owner gets the file, with safe headers\"\n    sessions, owners = _setup()\n    status, headers, body = download(_req(\"s-ada-123\", \"ada/report 2026 (final).pdf\"), sessions, owners, \"store\", [].append)\n    assert status == 200 and body == b\"%PDF ada\"\n    assert headers == {\"Content-Type\": \"application/octet-stream\", \"X-Content-Type-Options\": \"nosniff\",\n                       \"Content-Disposition\": 'attachment; filename=\"report_2026__final_.pdf\"'}\n\ndef test_refusals():\n    \"every refusal\"\n    sessions, owners = _setup()\n    logged = []\n    def call(session, file):\n        return download(_req(session, file), sessions, owners, \"store\", logged.append)[::2]\n    assert call(None, \"bob/notes.txt\") == (401, b\"log in first\")\n    assert call(\"forged\", \"bob/notes.txt\") == (401, b\"log in first\")\n    assert call(\"s-ada-123\", None) == (400, b\"which file?\") and call(\"s-ada-123\", \"\") == (400, b\"which file?\")\n    assert call(\"s-ada-123\", \"bob/notes.txt\") == (404, b\"not found\")\n    assert call(\"s-ada-123\", \"carol/plans.txt\") == (404, b\"not found\")\n    assert call(\"s-ada-123\", \"ada/missing.txt\") == (404, b\"not found\")\n    assert call(\"s-ada-123\", \"../secret.env\") == (403, b\"not allowed\")\n    assert call(\"s-bob-456\", \"bob/../../secret.env\\nINFO all good\") == (403, b\"not allowed\")\n    assert call(\"s-ada-123\", \".\") == (403, b\"not allowed\")\n    assert logged == [\"blocked traversal by ada: '../secret.env'\", \"blocked traversal by bob: 'bob/../../secret.env\\\\nINFO all good'\",\n                      \"blocked traversal by ada: '.'\"]\n    assert all(\"\\n\" not in line for line in logged)\n\ndef test_compare():\n    \"sessions are compared in constant time\"\n    import hmac\n    sessions, owners = _setup()\n    real, calls = hmac.compare_digest, []\n    def spy(a, b):\n        calls.append((a, b))\n        return real(a, b)\n    hmac.compare_digest = spy\n    try:\n        download(_req(\"s-bob-456\", \"bob/notes.txt\"), sessions, owners, \"store\", [].append)\n    finally:\n        hmac.compare_digest = real\n    assert calls, \"use hmac.compare_digest to check the session id\"")

## Warm-up

A user asked for a file by name, and two of those names reach outside the folder they were meant to stay in. `base / "/etc/shadow"` is especially sneaky: joining an **absolute** path throws the base away. Resolving the path and then checking it's still inside the base is the standard defence, and it's one of today's exercises.

## Thinking like an attacker

Security starts with three questions, a small **threat model**:

1. **What are you protecting?** The **assets**: users' data, passwords, money, the server itself, your reputation.
2. **Who might attack, and how?** Anyone on the internet; a logged-in user trying to reach someone else's data; a malicious file uploaded or a link clicked; a compromised dependency (Thursday).
3. **Where does untrusted data come in?** The **trust boundaries**: every place data crosses from somewhere you don't control into your code. Request bodies, query strings, headers and cookies; uploaded files and their **names**; command-line arguments and environment variables in shared systems; files from other people; responses from other services, including ones you call; even your own database, if anyone else can write to it.

Data that has crossed a boundary is **untrusted** until it's been checked, and checking it once, at the boundary, is far easier than remembering to be careful everywhere it goes.

A handful of principles cover most of the ground:

- **Least privilege**: give code, accounts and keys only the access they need (a report job needs to read the database, not drop tables).
- **Defence in depth**: assume one layer will fail, and have another (validation, then parameterised queries, then a database user without delete rights).
- **Secure defaults**: the safe behaviour should be what happens when nobody configures anything (debug off, new users without admin rights, everything denied unless allowed).
- **Fail closed**: when a check can't be completed (the permission service is down, a token can't be parsed), deny.
- **Keep it simple**: complicated security code has more places for bugs; use well-tested standard tools.

The **OWASP Top 10** (from the Open Worldwide Application Security Project) lists the most common serious web risks; broken access control (users reaching what isn't theirs) and injection (tomorrow) have been near the top for years. It's a good checklist for code review (Week 74).

## Validating input

**Validate with allow-lists, not deny-lists.** Saying what's **allowed** (a username is 3 to 20 letters, digits and underscores) is safe by default: anything you didn't think of is refused. Trying to list what's **forbidden** (no `<script>`, no `'`, no `../`) always misses something: an encoding, a capital letter, a Unicode look-alike.

Good validation, at the boundary:

- checks the **type** (a number is a number, not a string that looks like one);
- checks **ranges and lengths**, both minimum and maximum (a 10 MB "name" is an attack on your memory and your database);
- checks **formats** with full matches (`re.fullmatch`, not `re.match` or `re.search`, which accept extra text after a match);
- checks **choices** against a fixed set;
- **refuses unknown fields** instead of passing a whole request body to an update, so a user can't set `"is_admin": true` on themselves (a **mass assignment** attack);
- **normalises** before checking (strip spaces, a consistent case for emails, Unicode normalisation with `unicodedata.normalize("NFKC", text)` so look-alike characters can't sneak past);
- returns **clear errors** for honest mistakes, and never "fixes" bad input silently.

Note the `bool` check: in Python, `True` is an `int`, so `isinstance(True, int)` is `True` and a careless check would accept `"age": true` as `1`.

## Dangerous operations

Some operations turn **data into action**, and they're where untrusted input does the most damage:

| Danger | Example | Instead |
|---|---|---|
| **Path traversal** | `open(UPLOADS / request_name)` with `../../.ssh/id_rsa` | resolve, then check `is_relative_to(base)`; or look names up in a table of allowed files |
| **Evaluating text as code** | `eval(user_text)`, `exec`, `compile` | `json.loads`, or `ast.literal_eval` for Python literals only |
| **Unpickling** | `pickle.loads(cookie)` | JSON; `pickle` runs code chosen by whoever made the bytes |
| **Unsafe YAML** | `yaml.load(text)` with the full loader | `yaml.safe_load` |
| **Shell commands** | `os.system("convert " + filename)` | tomorrow: `subprocess.run([...])` with a list and no shell |
| **Unbounded work** | reading a whole upload into memory, a regular expression that backtracks for minutes | size limits, timeouts, simple patterns |

The rule behind every row: **never let untrusted data choose what code runs, or which file is touched**, without checking it against what's allowed.

## Failing safely

**Fail closed.** A permission check that throws an exception must end in "no", never "yes". Write checks so that the only path to "allowed" is the explicit one, and an `except` never returns success.

**Errors that don't leak.** A stack trace, an SQL error or a file path in an error page tells an attacker how your system is built, and exception messages often contain secrets (a database URL with its password, a token in a request). Show the user a short generic message and an **incident id**; log the full details, with the same id, where only you can read them.

**Logs that don't leak.** Logs are copied to monitoring services, read by many people and kept for years. Never log passwords, tokens, API keys, session cookies or full card numbers; **redact** them before they're written (Wednesday covers secrets in depth). And don't log raw user input unescaped: a newline in a username can forge a fake log line (log injection, tomorrow).

**Constant-time comparison.** Checking a secret with `==` stops at the first wrong character, so a careful attacker timing many requests can learn a token one character at a time. `hmac.compare_digest(a, b)` takes the same time whatever the inputs, and it's what to use for tokens, signatures and API keys (Week 77).

**Slow down guessing.** Limit failed logins per account and per address, with a lockout or growing delays, and give the **same** message whether the username or the password was wrong, so attackers can't discover which accounts exist.

### On your own computer

Static analysers find many of these mistakes automatically: **Bandit** (`pip install bandit`, then `bandit -r src/`) flags `eval`, `pickle`, `shell=True`, weak hashes and hard-coded passwords, and **Ruff** includes many of the same checks (its `S` rules). Run them in CI (Week 74) so new code is checked on every change.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a small program you've written in this course that takes input (the `spend` tracker from Week 25 is a good one) and draw its trust boundaries: where does outside data enter, and is it checked there? Tomorrow: **injection attacks**, where untrusted data is mistaken for code: SQL, shell commands, HTML and logs.